Mempersiapan: SparkSession & Dataset

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [14]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


A. Feature Engineering

In [15]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Membuat SparkSession
spark = SparkSession.builder.appName("Tugas11").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset CSV
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

# Menampilkan data
df.show(5)
df.printSchema()

# Membagi data menjadi training dan testing
train_data, test_data = df.randomSplit([0.75, 0.25], seed=101)

# Mengubah kolom kategori menjadi angka
indexer_menu = StringIndexer(
    inputCol="nama_menu",
    outputCol="menu_index",
    handleInvalid="keep"
)

indexer_cabang = StringIndexer(
    inputCol="cabang",
    outputCol="cabang_index",
    handleInvalid="keep"
)

indexer_waktu = StringIndexer(
    inputCol="waktu_pesan",
    outputCol="waktu_index",
    handleInvalid="keep"
)

# Melatih StringIndexer menggunakan data training
menu_model = indexer_menu.fit(train_data)
cabang_model = indexer_cabang.fit(train_data)
waktu_model = indexer_waktu.fit(train_data)

# Menerapkan hasil indexing pada data training dan testing
train_indexed = menu_model.transform(train_data)
train_indexed = cabang_model.transform(train_indexed)
train_indexed = waktu_model.transform(train_indexed)

test_indexed = menu_model.transform(test_data)
test_indexed = cabang_model.transform(test_indexed)
test_indexed = waktu_model.transform(test_indexed)

# Menggabungkan seluruh fitur menjadi satu kolom
assembler = VectorAssembler(
    inputCols=[
        "qty",
        "harga_satuan",
        "menu_index",
        "cabang_index",
        "waktu_index"
    ],
    outputCol="features"
)

train_final = assembler.transform(train_indexed)
test_final = assembler.transform(test_indexed)

# Menampilkan hasil feature engineering
train_final.select("nama_menu", "cabang", "waktu_pesan",
                   "qty", "harga_satuan", "features").show(5)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



+---------+--------+-----------+---+------------+--------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|            features|
+---------+--------+-----------+---+------------+--------------------+
|Americano|Magelang|      Malam|  1|       15000|[1.0,15000.0,0.0,...|
|Americano|Magelang|      Malam|  1|       20000|[1.0,20000.0,0.0,...|
|Americano|Magelang|      Malam|  1|       22000|[1.0,22000.0,0.0,...|
|Americano|Magelang|      Malam|  1|       25000|[1.0,25000.0,0.0,...|
|Americano|Magelang|      Malam|  1|       25000|[1.0,25000.0,0.0,...|
+---------+--------+-----------+---+------------+--------------------+
only showing top 5 rows



B. Train/Test Split & Pelatihan Model

In [16]:
from pyspark.ml.regression import LinearRegression

# Membuat model Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan"
)

# Melatih model menggunakan data training
model = lr.fit(train_final)

print("Model Linear Regression berhasil dilatih!")

# Melakukan prediksi pada data testing
predictions = model.transform(test_final)

# Menampilkan hasil prediksi
predictions.select(
    "total_penjualan",
    "prediction"
).show(10)

Model Linear Regression berhasil dilatih!
+------------------+------------------+
|   total_penjualan|        prediction|
+------------------+------------------+
|19721.170446051627|31588.286776672234|
|  21236.0956379149|31588.286776672234|
|  85834.8564401191|  88658.1012101797|
|102174.09100317833|  88658.1012101797|
|113044.63431555482|112868.08792928922|
|115810.29600214415|110707.62102790417|
|140386.40273615732|119786.36604757026|
| 73842.64420790318| 68433.12368093252|
|32345.642983082646|  45088.9184003266|
| 45510.16712269087| 67138.43821805112|
+------------------+------------------+
only showing top 10 rows



C. Evaluasi Model 

In [17]:
from pyspark.ml.evaluation import RegressionEvaluator

# Menghitung RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(predictions)

# Menghitung R2
evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(predictions)

# Menampilkan hasil evaluasi
print("Hasil Evaluasi Model Utama")
print("RMSE :", rmse)
print("R2   :", r2)

Hasil Evaluasi Model Utama
RMSE : 10885.996989766023
R2   : 0.9109215940092599


D. Eksperimen Perbandingan Fitur

In [18]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Menggabungkan fitur qty dan harga_satuan saja
assembler_basic = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features"
)

# Membuat fitur untuk data training dan testing
train_basic = assembler_basic.transform(train_data)
test_basic = assembler_basic.transform(test_data)

# Membuat model Linear Regression kedua
lr_basic = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan"
)

# Melatih model kedua
model_basic = lr_basic.fit(train_basic)

# Melakukan prediksi
predictions_basic = model_basic.transform(test_basic)

# Menghitung R2 model kedua
evaluator_basic = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2_basic = evaluator_basic.evaluate(predictions_basic)

# Membandingkan R2 kedua model
print("Perbandingan R2 Model")
print("Model Utama (5 fitur) :", r2)
print("Model Kedua (2 fitur) :", r2_basic)

# Menentukan model dengan R2 lebih tinggi
if r2 > r2_basic:
    print("Model utama memiliki kemampuan prediksi lebih baik.")
elif r2 < r2_basic:
    print("Model kedua memiliki kemampuan prediksi lebih baik.")
else:
    print("Kedua model memiliki nilai R2 yang sama.")

Perbandingan R2 Model
Model Utama (5 fitur) : 0.9109215940092599
Model Kedua (2 fitur) : 0.9107572632054387
Model utama memiliki kemampuan prediksi lebih baik.


E. Prediksi Skenario Baru

In [20]:
# Membuat 3 data pesanan baru
data_baru = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 18000),
    ("Americano", "Yogyakarta", "Siang", 3, 20000),
    ("Cappuccino", "Semarang", "Malam", 4, 25000)
]

kolom = [
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan"
]

df_baru = spark.createDataFrame(data_baru, kolom)

# Menampilkan data baru
print("Data Pesanan Baru")
df_baru.show()

# Mengubah data kategori menggunakan model StringIndexer sebelumnya
data_indexed = menu_model.transform(df_baru)
data_indexed = cabang_model.transform(data_indexed)
data_indexed = waktu_model.transform(data_indexed)

# Menggabungkan fitur menggunakan VectorAssembler yang sama
data_features = assembler.transform(data_indexed)

# Melakukan prediksi menggunakan model utama
hasil_prediksi = model.transform(data_features)

# Menampilkan hasil prediksi
print("Hasil Prediksi Total Penjualan")

hasil_prediksi.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "prediction"
).show(truncate=False)

Data Pesanan Baru
+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Kopi Susu|  Magelang|       Pagi|  2|       18000|
| Americano|Yogyakarta|      Siang|  3|       20000|
|Cappuccino|  Semarang|      Malam|  4|       25000|
+----------+----------+-----------+---+------------+

Hasil Prediksi Total Penjualan
+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31894.71790538232|
|Americano |Yogyakarta|Siang      |3  |20000       |60687.50231653663|
|Cappuccino|Semarang  |Malam      |4  |25000       |97047.65045135112|
+----------+----------+-----------+---+------------+-----------------+



In [ ]:
EXPLORASI

1. Membandingkan Hasil Prediksi dan Nilai Aktual

In [23]:
# Membandingkan nilai aktual dan hasil prediksi
hasil_perbandingan = predictions.select(
    col("total_penjualan").alias("penjualan_aktual"),
    col("prediction").alias("hasil_prediksi")
)

hasil_perbandingan.show(truncate=False)

+------------------+------------------+
|penjualan_aktual  |hasil_prediksi    |
+------------------+------------------+
|19721.170446051627|31588.286776672234|
|21236.0956379149  |31588.286776672234|
|85834.8564401191  |88658.1012101797  |
|102174.09100317833|88658.1012101797  |
|113044.63431555482|112868.08792928922|
|115810.29600214415|110707.62102790417|
|140386.40273615732|119786.36604757026|
|73842.64420790318 |68433.12368093252 |
|32345.642983082646|45088.9184003266  |
|45510.16712269087 |67138.43821805112 |
|85362.59400453049 |82269.6799174946  |
|95663.3347901574  |88322.176597272   |
|97390.04610730882 |88322.176597272   |
|99994.17567791023 |97400.9216169381  |
|130758.48098025571|112532.16331638153|
|73887.87471263618 |89187.95803577562 |
|116322.79570298787|110371.69641499648|
|127726.10967197592|119450.44143466257|
|19477.97002961815 |10740.473010359041|
|19799.429587480616|22845.46636991382 |
+------------------+------------------+
only showing top 20 rows



2. Eksplorasi Rata-Rata Jumlah Pembelian

In [31]:
from pyspark.sql.functions import avg

eksplorasi_qty = df.groupBy("nama_menu").agg(
    avg("qty").alias("rata_rata_pembelian"),
    sum("qty").alias("total_produk_terjual")
).orderBy(col("total_produk_terjual").desc())

print("Rata-Rata Pembelian Setiap Menu:")
eksplorasi_qty.show()

Rata-Rata Pembelian Setiap Menu:
+--------------+-------------------+--------------------+
|     nama_menu|rata_rata_pembelian|total_produk_terjual|
+--------------+-------------------+--------------------+
|     Americano| 2.9953917050691246|                1300|
|    Cappuccino|  3.064439140811456|                1284|
|     Kopi Susu|  2.960093896713615|                1261|
|        Es Teh| 3.0220588235294117|                1233|
|    Roti Bakar|                3.0|                1221|
|Kentang Goreng|  2.894088669950739|                1175|
+--------------+-------------------+--------------------+



In [33]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
